# Data Model Installer

Install a Databricks **Industry Data Model** (catalog, schemas, tables, foreign keys,
governance tags, and metric views) into a Unity Catalog catalog of your choice. Model
definitions come live from the
[lakehouse-industry-data-models](https://github.com/databricks-industry-solutions/lakehouse-industry-data-models)
repo, or from a local folder you provide.

**Databricks Serverless compatible** - every operation is a plain `spark.sql` call,
with Python threads for parallelism. No SparkContext, RDDs, caching, or
classic-compute-only APIs.

## Minimal use

**Select an industry in the `1. industry` widget and click `Run All`.** Everything else
has a sensible default (`size = mvm`, catalog = the industry name, 32 threads). On the
very first Run All nothing is selected yet, so the notebook asks you to pick an industry
and Run All again.

## How it runs (launcher pattern)

When you Run All interactively, the notebook **launches the install as a Databricks job**, prints the run URL, then **waits for the
job to finish**, pulsing a liveness line every 5s and pointing at the job URL so you can
monitor it closely. When the job completes it prints the **Catalog Explorer URL**. The job runs the install and tags
itself (prefix `dbx_vibe_agent_installer_`) with the `industry`, `size`, `version`, and
final `duration`. The launched job always runs the install in-place (it never
re-launches itself).

The installer resolves the **latest version** of the model, downloads its SQL, rewrites
the catalog name to yours, splits it into statements, and applies them in dependency
order with live timestamped progress:
`catalog -> schemas -> tables -> foreign keys -> tags -> metric views`.
Failures are captured and retried serially at the end, so transient concurrency errors
self-heal and re-running is safe (idempotent).

> `mvm` = *Minimal Viable Model* (recommended). `ecm` = *Enterprise Canonical Model* (full).

## Parameters (widgets)

Four widgets, shown in order:

| Widget | Meaning |
|---|---|
| **1. industry** | Industry to install (40 choices). Defaults to a placeholder so you choose explicitly. |
| **2. size** | `mvm` (minimal, recommended) or `ecm` (full). |
| **3. catalog** | Target catalog. Blank = the industry name. |
| **4. local install** | Optional local/Volume folder. If set, SQL is read from here and the GitHub download is skipped (use it to install an older version). |

Advanced settings are not shown as widgets and use built-in defaults, forwarded to the
launched job automatically: **32 threads x 20-statement batches** (the measured
serverless optimum), metric views **on**, source = this repo @ `main`.

### Installing an older version (local_install)

The repo serves the latest version. To install an older one: download that version's
folder (e.g. `data-models/<industry>/v2/<model_size>/`, containing `schemas/` and
`metrics/`), upload it to a Unity Catalog **Volume**, and set **local_install** to that
path (the model folder or its parent version folder; `schemas/` is located
automatically). When set, version auto-detection and the GitHub download are skipped.


In [ ]:
# === Widgets (creates the UI parameters) ===
INDUSTRIES = ["advertising", "agriculture", "airlines", "apparel_fashion", "automotive", "banking", "chemical_mfg", "clinical_trials", "construction", "consumer_goods", "ecommerce", "education", "energy_utilities", "food_beverage", "gaming", "genomics_biotech", "grocery", "health_insurance", "healthcare", "legal", "life_insurance", "manufacturing", "media_broadcasting", "mining", "ngo", "oil_gas", "payments_fintech", "pharmaceuticals", "real_estate", "restaurants", "retail", "semiconductors", "shipping_ports", "sports_entertainment", "staffing_hr", "telecommunication", "transport_shipping", "travel_hospitality", "waste_management", "water_utilities"]   # the 40 industries shipped in the repo

SELECT_PROMPT = "(select an industry)"   # forces an explicit choice on the first run

# Only FOUR widgets are shown, in order. Everything else (threads, batch size, metric
# views, source repo/ref, session id) uses the defaults in INSTALLER_DEFAULTS below and
# is forwarded automatically to the launched job, so the UI stays minimal.
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("model", SELECT_PROMPT, [SELECT_PROMPT] + INDUSTRIES, "1. industry")
dbutils.widgets.dropdown("model_size", "mvm", ["mvm", "ecm"], "2. size")
dbutils.widgets.text("catalog_name", "", "3. catalog (blank = industry name)")
dbutils.widgets.text("local_install", "", "4. local install (blank = pull from repo)")

# Advanced settings - intentionally NOT shown as widgets. 32 threads x 20-statement
# batches is the measured serverless optimum; metric views on; source = this repo @ main.
INSTALLER_DEFAULTS = {
    "threads": "32",
    "batch_size": "20",
    "include_metrics": "true",
    "source_repo": "databricks-industry-solutions/lakehouse-industry-data-models",
    "source_ref": "main",
    "github_token": "",
}


In [ ]:
# === Shared helpers ===
import datetime
import threading
import os
import json

INSTALLER_TAG_PREFIX = "dbx_vibe_agent_installer_"

_LOG_LOCK = threading.Lock()
_LOG_BUFFER = []   # mirrored to a UC Volume file so logs are retrievable after the run
_SINK = {"path": None}   # set by setup_log_sink(); log() appends to it inline


def log(msg):
    """Print a timestamped log line (streams live in the cell) and append it to the
    Volume log sink INLINE on the calling thread.

    Inline append (vs a background flush daemon) is deliberate: during a heavy
    spark.sql phase the 32 worker threads + blocked main thread starve any daemon
    thread of the GIL, so a daemon-based flush would not update the file until the
    phase ended. Worker threads call log() between gRPC calls (they hold the GIL
    then), so inline append keeps the file genuinely live during the phase."""
    line = "[%s] %s" % (datetime.datetime.now().strftime("%H:%M:%S"), msg)
    with _LOG_LOCK:
        _LOG_BUFFER.append(line)
        print(line, flush=True)
        path = _SINK["path"]
        if path:
            # Rewrite the whole buffer (truncate+write). UC Volume FUSE does not
            # surface "a" (append) writes to a separate `fs cp` reader, but a full
            # "w" rewrite IS visible, so this keeps the CLI tail genuinely live.
            # Single serialized writer (under _LOG_LOCK) -> always a consistent file.
            try:
                with open(path, "w") as f:
                    f.write("\n".join(_LOG_BUFFER) + "\n")
            except Exception:
                pass


def _flush_log_durable():
    """Persist the full log buffer durably before dbutils.notebook.exit(). The per-line
    open('w') rewrites go through UC Volume FUSE, whose write-back is async, so the
    final rewrite (the one carrying the verdict) can be dropped when the process
    terminates even after os.fsync(). dbutils.fs.put writes synchronously through the
    Volumes API, so the verdict always survives; open('w') is kept for live-tail."""
    with _LOG_LOCK:
        path = _SINK["path"]
        if not path:
            return
        content = "\n".join(_LOG_BUFFER) + "\n"
        try:
            with open(path, "w") as f:
                f.write(content)
                f.flush()
                os.fsync(f.fileno())
        except Exception:
            pass
        try:
            dbutils.fs.put(path, content, True)
        except Exception:
            pass


In [ ]:
# === Core SQL logic (validated offline against the real repo SQL) ===
"""Core SQL-processing logic for data-model-installer.ipynb.

Authored here as a standalone module so it can be unit-tested locally against
real repo SQL before being embedded verbatim into the notebook cells.
No Spark / Databricks dependency in this module.
"""
import re
from collections import OrderedDict


def split_sql(text):
    """Split a SQL script into individual statements on top-level ';'.

    Aware of: -- line comments, /* */ block comments, '...' single-quoted
    strings (with '' escaping), `...` backtick identifiers, and $$...$$
    dollar-quoted blocks (used by metric views' YAML body). Comments are
    stripped from the emitted statements; string/identifier/dollar bodies
    are preserved verbatim (including any ';' inside them).
    """
    stmts = []
    buf = []
    i = 0
    n = len(text)
    in_line_comment = False
    in_block_comment = False
    in_squote = False
    in_btick = False
    in_dollar = False
    while i < n:
        c = text[i]
        nxt = text[i + 1] if i + 1 < n else ''
        if in_line_comment:
            if c == '\n':
                in_line_comment = False
                buf.append(c)
            i += 1
            continue
        if in_block_comment:
            if c == '*' and nxt == '/':
                in_block_comment = False
                i += 2
                continue
            i += 1
            continue
        if in_dollar:
            if c == '$' and nxt == '$':
                buf.append('$$')
                in_dollar = False
                i += 2
                continue
            buf.append(c)
            i += 1
            continue
        if in_squote:
            buf.append(c)
            if c == "'":
                if nxt == "'":
                    buf.append(nxt)
                    i += 2
                    continue
                in_squote = False
            i += 1
            continue
        if in_btick:
            buf.append(c)
            if c == '`':
                in_btick = False
            i += 1
            continue
        # not inside any quote/comment
        if c == '-' and nxt == '-':
            in_line_comment = True
            i += 2
            continue
        if c == '/' and nxt == '*':
            in_block_comment = True
            i += 2
            continue
        if c == '$' and nxt == '$':
            in_dollar = True
            buf.append('$$')
            i += 2
            continue
        if c == "'":
            in_squote = True
            buf.append(c)
            i += 1
            continue
        if c == '`':
            in_btick = True
            buf.append(c)
            i += 1
            continue
        if c == ';':
            stmt = ''.join(buf).strip()
            if stmt:
                stmts.append(stmt)
            buf = []
            i += 1
            continue
        buf.append(c)
        i += 1
    tail = ''.join(buf).strip()
    if tail:
        stmts.append(tail)
    return stmts


def detect_catalog_token(catalogs_sql):
    """Return the backtick-quoted catalog identifier declared by CREATE CATALOG."""
    m = re.search(r"CREATE\s+CATALOG\s+(?:IF\s+NOT\s+EXISTS\s+)?`([^`]+)`",
                  catalogs_sql, re.IGNORECASE)
    return m.group(1) if m else None


def rewrite_catalog(text, src_token, dst_token):
    """Replace every backtick-quoted occurrence of the source catalog token."""
    if not src_token or src_token == dst_token:
        return text
    return text.replace("`%s`" % src_token, "`%s`" % dst_token)


def categorize(stmt):
    """Classify a statement for ordered, phased execution."""
    u = re.sub(r"\s+", " ", stmt).strip().upper()
    if u.startswith("CREATE CATALOG"):
        return "catalog"
    if u.startswith("CREATE DATABASE") or u.startswith("CREATE SCHEMA"):
        return "schema"
    if u.startswith("CREATE OR REPLACE TABLE") or u.startswith("CREATE TABLE") \
            or u.startswith("CREATE EXTERNAL TABLE"):
        return "table"
    if "ADD CONSTRAINT" in u and "FOREIGN KEY" in u:
        return "fk"
    if "SET TAGS" in u or "UNSET TAGS" in u:
        return "tag"
    if ("CREATE OR REPLACE VIEW" in u or u.startswith("CREATE VIEW")
            or "CREATE MATERIALIZED VIEW" in u):
        return "metric"
    return "other"


_TARGET_RE = re.compile(
    r"ALTER\s+(?:TABLE|SCHEMA|VIEW|MATERIALIZED\s+VIEW)\s+(`[^`]+`(?:\.`[^`]+`){0,2})",
    re.IGNORECASE)


def target_key(stmt):
    """Best-effort target object key for grouping same-object mutations.

    For ALTER ... we group by the table/schema (first two parts) so all FKs
    and column tags for one table stay contiguous in one batch, avoiding
    concurrent-modification conflicts when run across threads.
    """
    m = _TARGET_RE.search(stmt)
    if not m:
        return ""
    parts = m.group(1).split("`.`")
    cleaned = [p.strip("`") for p in parts]
    return ".".join(cleaned)


def build_batches(statements, batch_size, group_by_target=False):
    """Chunk statements into batches.

    When group_by_target is True, every statement that mutates the same target
    object becomes ONE batch (one batch per distinct table/schema). Each batch
    runs sequentially on a single thread, so two threads never mutate the same
    table concurrently (no Delta concurrent-modification conflicts), while
    different tables run fully in parallel. One-batch-per-table also gives the
    thread pool the finest grain for load-balancing, so a hub table with many
    FKs/tags cannot monopolize a coarse multi-table batch. Statements with no
    detectable target share a single fallback batch.

    When group_by_target is False (independent statements, e.g. CREATE TABLE on
    distinct tables), statements are chunked into fixed groups of batch_size.
    """
    if not group_by_target:
        return [statements[i:i + batch_size]
                for i in range(0, len(statements), batch_size)]
    groups = OrderedDict()
    for s in statements:
        k = target_key(s)
        groups.setdefault(k, []).append(s)
    return [g for g in groups.values()]


_SET_TAGS_RE = re.compile(
    r"^(?P<prefix>.*?\bSET\s+TAGS\s*\()(?P<body>.*)\)\s*$",
    re.IGNORECASE | re.DOTALL)


def merge_tag_statements(tag_stmts):
    """Merge multiple `SET TAGS` on the SAME target (and same column) into one
    statement, so one metadata commit sets all of that target's tags instead of
    one commit per tag. Grouping key is the full text up to `SET TAGS (` (which
    includes the table and any `ALTER COLUMN c`), so only tags on the identical
    target+column are combined. `UNSET TAGS` and any non-matching statement pass
    through unchanged, preserving order relative to the merged block.
    """
    merged = OrderedDict()
    passthrough = []
    for s in tag_stmts:
        m = _SET_TAGS_RE.match(s.strip())
        if not m:
            passthrough.append(s)
            continue
        prefix = re.sub(r"\s+", " ", m.group("prefix")).strip()
        body = m.group("body").strip()
        merged.setdefault(prefix, []).append(body)
    out = ["%s%s)" % (prefix, ", ".join(bodies)) for prefix, bodies in merged.items()]
    return out + passthrough


In [ ]:
# === JobLauncher: launch this notebook as a tagged Databricks job (agent pattern) ===
import re as _jl_re


class JobLauncher:
    """Create/reuse a named Databricks job for this notebook and run it, with tags.

    Mirrors the agent's launcher: build with the notebook path, the widget values to
    pass as base_parameters, and a {tag_key: tag_value} dict attached to the job.
    """

    _TAG_SAFE_RE = _jl_re.compile(r"[^A-Za-z0-9._-]")

    @staticmethod
    def _sanitize_tag(value):
        s = JobLauncher._TAG_SAFE_RE.sub("_", str(value))
        s = _jl_re.sub(r"_+", "_", s)
        return s.strip("_").strip(".").strip("-")

    def __init__(self, notebook_path, widget_key_values, job_tags=None):
        self.notebook_path = str(notebook_path)
        self.widget_key_values = {str(k): str(v) for k, v in widget_key_values.items()}
        raw = dict(job_tags or {})
        self.job_tags = {self._sanitize_tag(k): self._sanitize_tag(v) for k, v in raw.items()}

    @staticmethod
    def _detect_compute_type():
        """Detect serverless vs classic (ported from the agent's proven launcher).

        Serverless compute exposes a non-empty `clusterUsageTags.clusterId`, so a
        clusterId-only check misfires and wrongly attaches a classic cluster - which a
        serverless-only workspace rejects ("Only serverless compute is supported").
        The reliable signals are the IS_SERVERLESS env var and the absence of a
        cluster *name*. Returns (is_serverless, cluster_id)."""
        import os as _jl_os
        if _jl_os.environ.get("IS_SERVERLESS", "").upper() == "TRUE":
            return True, None
        try:
            spark.conf.get("spark.databricks.clusterUsageTags.clusterName")
        except Exception:
            return True, None
        try:
            _cid = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
            if _cid:
                return False, _cid
        except Exception:
            pass
        return True, None

    @staticmethod
    def _get_workspace_context():
        host, org = "", ""
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            try:
                host = ctx.apiUrl().get()
            except Exception:
                pass
            try:
                org = ctx.workspaceId().get()
            except Exception:
                pass
        except Exception:
            pass
        if not host:
            try:
                from databricks.sdk import WorkspaceClient as _WC
                host = str(_WC().config.host)
            except Exception:
                pass
        return (host or "").rstrip("/"), org

    @staticmethod
    def get_current_notebook_path():
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            try:
                p = ctx.notebookPath().get()
                if p:
                    return p
            except Exception:
                pass
            import json as _j
            c = _j.loads(ctx.toJson())
            for k in ("notebook_path", "notebookPath"):
                v = (c.get("extraContext") or {}).get(k, "") or (c.get("tags") or {}).get(k, "")
                if v:
                    return v
        except Exception:
            pass
        return ""

    def launch(self, job_name=None, run_name=None):
        import time as _t
        from databricks.sdk import WorkspaceClient as _WC
        from databricks.sdk.service import jobs as _jobs
        out = {"success": False, "job_id": None, "run_id": None, "job_url": "", "error": None}
        try:
            w = _WC()
            job_name = job_name or ("dbx_vibe_installer_%d" % int(_t.time()))
            is_serverless, cluster_id = self._detect_compute_type()

            def _build_task(attach_cluster):
                t = _jobs.Task(
                    task_key="install",
                    notebook_task=_jobs.NotebookTask(
                        notebook_path=self.notebook_path,
                        base_parameters=self.widget_key_values),
                    timeout_seconds=14400,
                )
                if attach_cluster and cluster_id:
                    t.existing_cluster_id = cluster_id
                return t

            existing = None
            try:
                for j in w.jobs.list(name=job_name):
                    if j.settings and j.settings.name == job_name:
                        existing = j.job_id
                        break
            except Exception:
                pass

            def _create_and_run(attach_cluster):
                task = _build_task(attach_cluster)
                if existing:
                    w.jobs.reset(job_id=existing,
                                 new_settings=_jobs.JobSettings(name=job_name, tags=self.job_tags, tasks=[task]))
                    jid = existing
                else:
                    jid = w.jobs.create(name=job_name, tags=self.job_tags, tasks=[task]).job_id
                r = w.jobs.run_now(job_id=jid)
                return jid, r

            try:
                job_id, run = _create_and_run(attach_cluster=(not is_serverless))
            except Exception as ce:
                # Defense in depth: if a serverless-only workspace rejects the attached
                # cluster (detection misfired), retry as a pure serverless task.
                if "serverless" in str(ce).lower() and not is_serverless:
                    job_id, run = _create_and_run(attach_cluster=False)
                else:
                    raise
            host, org = self._get_workspace_context()
            url = "%s/jobs/%s/runs/%s%s" % (host, job_id, run.run_id, ("?o=%s" % org if org else "")) if host else ""
            out.update({"success": True, "job_id": job_id, "run_id": run.run_id, "job_url": url})
        except Exception as e:
            out["error"] = str(e)
        return out

    @staticmethod
    def wait_for_run(run_id, job_url="", pulse_seconds=20, logger=None):
        """Block until the launched run reaches a terminal state, emitting a liveness
        timestamped pulse every pulse_seconds (no link). Returns
        {life_cycle_state, result_state, notebook_output, error}."""
        import time as _t
        from databricks.sdk import WorkspaceClient as _WC
        log = logger or (lambda m: print(m, flush=True))
        w = _WC()
        terminal = {"TERMINATED", "INTERNAL_ERROR", "SKIPPED"}
        out = {"life_cycle_state": "", "result_state": "", "notebook_output": "", "error": None}
        start = _t.time()
        while True:
            try:
                run = w.jobs.get_run(run_id=run_id)
            except Exception as e:
                log("   poll error (%s) - retrying in %ds" % (str(e)[:120], pulse_seconds))
                _t.sleep(pulse_seconds)
                continue
            state = run.state if run else None
            st = state.life_cycle_state.value if (state and state.life_cycle_state) else ""
            rs = state.result_state.value if (state and state.result_state) else ""
            elapsed = int(_t.time() - start)
            if st in terminal:
                out["life_cycle_state"], out["result_state"] = st, rs
                try:
                    trid = run.tasks[0].run_id if (run and run.tasks) else run_id
                    ro = w.jobs.get_run_output(run_id=trid)
                    out["notebook_output"] = (ro.notebook_output.result if ro.notebook_output else "") or ""
                    if ro.error:
                        out["error"] = ro.error
                except Exception as e:
                    out["error"] = str(e)
                return out
            log("   [%s] job still running [%s] %ds elapsed"
                % (_t.strftime("%H:%M:%S"), st or "PENDING", elapsed))
            _t.sleep(pulse_seconds)

    @staticmethod
    def update_job_tags(updated_tags):
        """Merge tags onto the job currently running this notebook (best effort)."""
        res = {"success": False, "error": None}
        if not updated_tags:
            res["error"] = "empty"
            return res
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            job_id_str = ""
            try:
                job_id_str = ctx.jobId().get()
            except Exception:
                pass
            if not job_id_str:
                res["error"] = "not running as a job (no jobId)"
                return res
            from databricks.sdk import WorkspaceClient as _WC
            from databricks.sdk.service import jobs as _jobs
            w = _WC()
            job_id = int(job_id_str)
            info = w.jobs.get(job_id=job_id)
            existing = dict(info.settings.tags or {})
            new = {JobLauncher._sanitize_tag(k): JobLauncher._sanitize_tag(v) for k, v in updated_tags.items()}
            merged = {**existing, **new}
            w.jobs.update(job_id=job_id, new_settings=_jobs.JobSettings(tags=merged))
            res["success"] = True
        except Exception as e:
            res["error"] = str(e)
        return res


In [ ]:
# === Config, model loading (repo or local), and plan building ===
DEFAULT_DDL_THREADS = 8   # UC-throttle-safe concurrency cap for fk/tag phases

import json as _json
import os
import re as _re
import urllib.request
from collections import OrderedDict


def _wget(name, default=""):
    """Read a widget/param value if present, else return default. Advanced settings are
    passed as job parameters (not shown as widgets), so they are read through this safe
    getter rather than dbutils.widgets.get, which raises when a widget was never created
    in an interactive run."""
    try:
        v = dbutils.widgets.get(name)
    except Exception:
        return default
    return v if (v is not None and v != "") else default


def resolve_config():
    """Read widgets into a config dict and log the resolved inputs."""
    industry = dbutils.widgets.get("model").strip()
    model_size = dbutils.widgets.get("model_size").strip()
    D = INSTALLER_DEFAULTS
    cfg = {
        "industry": industry,
        "model_size": model_size,
        "catalog": dbutils.widgets.get("catalog_name").strip() or industry,
        "local_install": dbutils.widgets.get("local_install").strip(),
        "session_id": _wget("session_id", "").strip(),
        "threads": int(_wget("threads", D["threads"]).strip() or D["threads"]),
        "batch_size": int(_wget("batch_size", D["batch_size"]).strip() or D["batch_size"]),
        "include_metrics": _wget("include_metrics", D["include_metrics"]).strip().lower() == "true",
        "source_repo": _wget("source_repo", D["source_repo"]).strip() or D["source_repo"],
        "source_ref": _wget("source_ref", D["source_ref"]).strip() or D["source_ref"],
        "github_token": _wget("github_token", D["github_token"]).strip(),
        "resolved_version": "unknown",
    }
    # UC throttles metadata DDL (ADD CONSTRAINT/SET TAGS) at high concurrency, so the
    # fk/tag phases run at a capped concurrency while object creation uses full threads.
    cfg["ddl_threads"] = max(4, min(cfg["threads"], DEFAULT_DDL_THREADS))
    assert industry in INDUSTRIES, "Unknown industry: %s" % industry
    assert model_size in ("mvm", "ecm"), "model_size must be mvm or ecm"
    assert cfg["threads"] >= 1 and cfg["batch_size"] >= 1
    cfg["mode"] = "LOCAL" if cfg["local_install"] else "REPO"
    return cfg


def _http_get(url, token=""):
    req = urllib.request.Request(url)
    req.add_header("Accept", "application/vnd.github+json")
    req.add_header("User-Agent", "data-model-installer")
    if token:
        req.add_header("Authorization", "Bearer %s" % token)
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read().decode("utf-8")


def _gh_contents(cfg, path):
    url = "https://api.github.com/repos/%s/contents/%s?ref=%s" % (
        cfg["source_repo"], path, cfg["source_ref"])
    return _json.loads(_http_get(url, cfg["github_token"]))


def latest_version(cfg):
    """Highest version folder (v1, v2, ...) available for the industry in the repo."""
    items = _gh_contents(cfg, "data-models/%s" % cfg["industry"])
    versions = sorted(
        (it["name"] for it in items
         if it.get("type") == "dir" and _re.match(r"^v\d+$", it["name"])),
        key=lambda v: int(v[1:]))
    if not versions:
        raise Exception("No version folders (vN) for industry %s" % cfg["industry"])
    return versions[-1]


def _resolve_repo_base(cfg):
    versions = _gh_contents(cfg, "data-models/%s" % cfg["industry"])
    vs = sorted((it["name"] for it in versions
                 if it.get("type") == "dir" and _re.match(r"^v\d+$", it["name"])),
                key=lambda v: int(v[1:]))
    if not vs:
        raise Exception("No version folders (vN) for industry %s" % cfg["industry"])
    cfg["resolved_version"] = vs[-1]
    log("Versions available: %s  -> using %s" % (vs, vs[-1]))
    return "data-models/%s/%s/%s" % (cfg["industry"], vs[-1], cfg["model_size"])


def _resolve_local_base(cfg):
    p = cfg["local_install"]
    cfg["resolved_version"] = "local"
    for cand in (p, os.path.join(p, cfg["model_size"])):
        if os.path.isdir(os.path.join(cand, "schemas")):
            return cand
    raise Exception("No 'schemas' folder under %r (tried direct and /%s). Point "
                    "local_install at the model folder containing schemas/ and "
                    "metrics/." % (p, cfg["model_size"]))


def _list_sql(ref, is_local, cfg):
    if is_local:
        if not os.path.isdir(ref):
            return []
        return [(n, os.path.join(ref, n)) for n in sorted(os.listdir(ref)) if n.endswith(".sql")]
    items = _gh_contents(cfg, ref)
    return [(it["name"], it["download_url"]) for it in items
            if it.get("type") == "file" and it["name"].endswith(".sql")]


def _read_sql(ref, cfg):
    if ref.startswith("http://") or ref.startswith("https://"):
        return _http_get(ref, cfg["github_token"])
    with open(ref, "r") as f:
        return f.read()


def _rank(name):
    if "catalog" in name:
        return (0, name)
    if "foreign_key" in name:
        return (2, name)
    return (1, name)


def build_plan(cfg):
    """Load the model SQL, rewrite the catalog, split, and categorize into phases."""
    is_local = bool(cfg["local_install"])
    if is_local:
        base = _resolve_local_base(cfg)
        schema_files = _list_sql(os.path.join(base, "schemas"), True, cfg)
        metric_files = _list_sql(os.path.join(base, "metrics"), True, cfg) if cfg["include_metrics"] else []
        log("Local base: %s" % base)
    else:
        base = _resolve_repo_base(cfg)
        schema_files = _list_sql(base + "/schemas", False, cfg)
        metric_files = _list_sql(base + "/metrics", False, cfg) if cfg["include_metrics"] else []
    schema_files.sort(key=lambda nv: _rank(nv[0]))
    metric_files.sort(key=lambda nv: nv[0])
    log("Schema files: %d   Metric files: %d" % (len(schema_files), len(metric_files)))

    src_token = None
    raw_by_name = {}
    for name, ref in schema_files + metric_files:
        raw = _read_sql(ref, cfg)
        raw_by_name[name] = raw
        if src_token is None and "catalog" in name:
            src_token = detect_catalog_token(raw)
    if src_token is None:
        for raw in raw_by_name.values():
            src_token = detect_catalog_token(raw)
            if src_token:
                break
    log("Source catalog token: %s  ->  target: %s" % (src_token, cfg["catalog"]))

    plan = OrderedDict((k, []) for k in ("catalog", "schema", "table", "fk", "tag", "metric", "other"))
    for name, ref in schema_files + metric_files:
        rewritten = rewrite_catalog(raw_by_name[name], src_token, cfg["catalog"])
        for stmt in split_sql(rewritten):
            plan[categorize(stmt)].append(stmt)

    # Merge multiple SET TAGS on the same target+column into one statement -> one
    # metadata commit instead of many (per-table tag commits are the long pole).
    n_tags_raw = len(plan["tag"])
    plan["tag"] = merge_tag_statements(plan["tag"])
    if n_tags_raw != len(plan["tag"]):
        log("Tag merge: %d -> %d statements" % (n_tags_raw, len(plan["tag"])))

    log("Execution plan:")
    total = 0
    for k, v in plan.items():
        if v:
            log("    %-8s %d" % (k, len(v)))
            total += len(v)
    log("    %-8s %d" % ("TOTAL", total))
    return plan


In [ ]:
# === Threaded batch executor with LIVE heartbeat progress + serial retry ===
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

_IGNORABLE = ("already exists", "already_exists")
# Transient UC/metastore server-side conditions. Under sustained concurrent DDL the
# Unity Catalog API throttles and returns 504/503/429 etc. These are NOT bad SQL -
# they self-heal on a short backoff, so we retry them inline instead of failing.
_TRANSIENT = (
    "uc_client_exception", "failed to contact the unity catalog",
    "gateway time", "504", "503", "502", "429", "too many requests",
    "deadline exceeded", "temporarily unavailable", "request timed out",
    "service unavailable", "connection reset", "concurrent")
PROGRESS_SECONDS = 3   # min seconds between live progress lines (worker-driven)
MAX_RETRIES = 6        # inline attempts per statement for transient UC errors


def _ignorable(msg):
    m = msg.lower()
    return any(tok in m for tok in _IGNORABLE)


def _catalog_exists(catalog):
    """True if the catalog is actually present (defensive check after CREATE)."""
    try:
        spark.sql("DESCRIBE CATALOG `%s`" % catalog)
        return True
    except Exception:
        return False


def _ensure_catalog(catalog, log):
    """Create the catalog robustly across metastore configurations.

    Plain `CREATE CATALOG` works when the metastore has a storage root. On
    workspaces where only Default Storage is enabled (no metastore storage root),
    that statement fails with "Metastore storage root URL does not exist" and
    Default Storage catalogs can ONLY be made in the UI. In that case we fall back
    to a real external location and create the catalog with an explicit MANAGED
    LOCATION under it. Industry-agnostic: external locations are discovered at
    runtime via SHOW EXTERNAL LOCATIONS, never hardcoded."""
    if _catalog_exists(catalog):
        return
    try:
        spark.sql("CREATE CATALOG IF NOT EXISTS `%s`" % catalog)
        if _catalog_exists(catalog):
            return
    except Exception as e:
        low = str(e).lower()
        needs_loc = ("storage root url does not exist" in low
                     or "default storage" in low
                     or "provide a storage location" in low
                     or "managed location" in low)
        if not needs_loc:
            raise
        log("   No metastore storage root on this workspace; discovering an external "
            "location to use as MANAGED LOCATION ...")
    cands = []
    try:
        for r in spark.sql("SHOW EXTERNAL LOCATIONS").collect():
            d = r.asDict()
            name = (d.get("name") or "").strip()
            url = (d.get("url") or "").strip().rstrip("/")
            if url and name and not name.startswith("__"):
                cands.append((name, url))
    except Exception as e:
        log("   Could not list external locations: %s" % str(e)[:160])
    last = None
    for name, url in cands:
        loc = "%s/%s" % (url, catalog)
        try:
            spark.sql("CREATE CATALOG IF NOT EXISTS `%s` MANAGED LOCATION '%s'"
                      % (catalog, loc))
            if _catalog_exists(catalog):
                log("   created catalog `%s` (MANAGED LOCATION under external location `%s`)"
                    % (catalog, name))
                return
        except Exception as e:
            last = str(e)
            continue
    raise Exception(
        "catalog `%s` could not be created. This workspace has no metastore storage "
        "root and Default Storage catalogs are UI-only. Grant an external location "
        "with CREATE MANAGED STORAGE (or create the catalog in the UI), then retry. "
        "Tried %d external location(s). Last error: %s"
        % (catalog, len(cands), (last or "no usable external location found")[:240]))


def _transient(msg):
    m = msg.lower()
    return any(tok in m for tok in _TRANSIENT)


# Author-defect errors in the SOURCE SQL (bad column / function arg / syntax /
# parameter). These are DETERMINISTIC: they fail identically on every attempt, so
# re-running them in retry_failed is pure wasted time (each metric-view recompile
# costs ~50s server-side). We detect them and skip the retry for those statements
# while still reporting them. UNRESOLVED_RELATION is deliberately EXCLUDED - a
# missing table can be a transient ordering artifact that a serial retry fixes.
_DETERMINISTIC = (
    "unresolved_column", "unresolved_routine", "unresolved_field",
    "unresolved_using_column", "parse_syntax_error", "invalid_parameter_value",
    "datatype_mismatch", "ambiguous_reference", "invalid_column",
    "cannot_resolve", "wrong_num_args", "unsupported_call",
)


def _deterministic(msg):
    m = (msg or "").lower()
    return any(tok in m for tok in _DETERMINISTIC)


def _exec_with_backoff(stmt):
    """Run one statement, retrying transient UC errors with exponential backoff +
    jitter. Returns (ok, err). Non-transient errors fail immediately."""
    import random
    delay = 0.5
    last = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            spark.sql(stmt)
            return True, None
        except Exception as e:
            last = str(e)
            if _ignorable(last) or not _transient(last) or attempt == MAX_RETRIES:
                return False, last
            time.sleep(delay + random.uniform(0, delay))
            delay = min(delay * 2, 16.0)
    return False, last


def run_phase(name, statements, threads, batch_size, group=False, serial=False):
    """Run one phase across threads with LIVE worker-driven progress.

    Progress is emitted by the WORKER threads (throttled to one line every
    PROGRESS_SECONDS), not by a separate daemon. A daemon heartbeat gets GIL-starved
    while 32 workers hammer spark.sql, so it only fires between phases. Workers hold
    the GIL between gRPC calls, so emitting from there keeps the log genuinely live.
    Returns [(stmt, err)]."""
    if not statements:
        log("[%s] nothing to do" % name)
        return []
    batches = build_batches(statements, batch_size, group_by_target=group)
    total = len(statements)
    workers = 1 if serial else threads
    prog = {"done": 0, "ok": 0, "ignored": 0, "failed": 0, "last": 0.0, "shown_err": False}
    failures = []
    lock = threading.Lock()
    t0 = time.time()
    prog["last"] = t0
    log("[%s] START %d statements in %d batches, %d workers" % (name, total, len(batches), workers))

    def _emit_progress():
        # caller MUST hold lock
        d, ok, ig, fa = prog["done"], prog["ok"], prog["ignored"], prog["failed"]
        pct = (100.0 * d / total) if total else 100.0
        log("[%s] %d/%d (%.0f%%) ok=%d ignored=%d failed=%d  %.0fs" % (
            name, d, total, pct, ok, ig, fa, time.time() - t0))

    def do_batch(batch):
        for stmt in batch:
            ok, err = _exec_with_backoff(stmt)   # plain spark.sql + transient backoff
            with lock:
                prog["done"] += 1
                if ok:
                    prog["ok"] += 1
                elif _ignorable(err):
                    prog["ignored"] += 1
                else:
                    prog["failed"] += 1
                    failures.append((stmt, err))
                    if not prog["shown_err"]:
                        prog["shown_err"] = True
                        log("[%s] first failure: %s" % (name, " ".join(err.split())[:200]))
                now = time.time()
                if now - prog["last"] >= PROGRESS_SECONDS and prog["done"] < total:
                    prog["last"] = now
                    _emit_progress()

    with ThreadPoolExecutor(max_workers=workers) as ex:
        for fut in as_completed([ex.submit(do_batch, b) for b in batches]):
            fut.result()

    log("[%s] DONE in %.1fs  ok=%d ignored=%d failed=%d" % (
        name, time.time() - t0, prog["ok"], prog["ignored"], prog["failed"]))
    return failures


def retry_failed(failures, passes=3):
    """Re-run failed statements serially; transient/ordering errors self-heal.

    Statements whose first-pass error is a deterministic SOURCE defect (bad column,
    bad function arg, syntax/param error) can NEVER succeed on retry, so we skip
    re-executing them entirely (saves ~50s each - a metric-view recompile cost) and
    keep them as failures. Only transient/ordering errors are retried."""
    deterministic = [(ph, st, er) for (ph, st, er) in failures if _deterministic(er)]
    remaining = [(ph, st, er) for (ph, st, er) in failures if not _deterministic(er)]
    if deterministic:
        log("Skipping retry for %d deterministic source defect(s) - they cannot "
            "self-heal (saved ~%ds)" % (len(deterministic), 50 * len(deterministic)))
    for p in range(1, passes + 1):
        if not remaining:
            break
        log("=== retry pass %d: %d statements ===" % (p, len(remaining)))
        still = []
        for phase, stmt, _ in remaining:
            ok, err = _exec_with_backoff(stmt)
            if ok or _ignorable(err or ""):
                continue
            still.append((phase, stmt, err))
        log("    after pass %d: %d remaining" % (p, len(still)))
        if len(still) == len(remaining):
            remaining = still
            break
        remaining = still
    return deterministic + remaining


## `main` - launch the job (or run the install)

If `session_id` is blank this cell launches the install as a Databricks job and prints
the run URL. When the job runs (session_id set) this same cell performs the install, so
all timestamped progress lands in this one cell's output. Earlier cells only define
functions and widgets.


In [ ]:
# === main: launcher gate, then the full install (all timestamped logs below) ===
def _gen_session_id():
    import uuid
    return str((uuid.uuid4().int >> 64) & 9223372036854775807)


def _running_as_job():
    """True when this notebook is executing inside a Databricks job run (jobId set).
    Hard backstop so a launched job never re-launches itself, even if its session_id
    job parameter is somehow unreadable."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return bool(ctx.jobId().get())
    except Exception:
        return False


def launch_and_wait(cfg):
    """Launch this notebook as a tagged Databricks job, then BLOCK until the job
    finishes, emitting a timestamped liveness pulse every 20s. On
    completion, surface the job result and the Catalog Explorer URL. Returns
    (handled, success, exit_value)."""
    sid = _gen_session_id()
    try:
        version = "local" if cfg["local_install"] else latest_version(cfg)
    except Exception:
        version = "unknown"
    # Forward the resolved config to the job as base_parameters. The four UI widgets
    # travel under their widget names; the advanced settings (not widgets) travel as
    # plain job params and are read back via _wget in resolve_config.
    widgets = {
        "model": cfg["industry"],
        "model_size": cfg["model_size"],
        "catalog_name": cfg["catalog"],
        "local_install": cfg["local_install"],
        "session_id": sid,
        "threads": str(cfg["threads"]),
        "batch_size": str(cfg["batch_size"]),
        "include_metrics": "true" if cfg["include_metrics"] else "false",
        "source_repo": cfg["source_repo"],
        "source_ref": cfg["source_ref"],
        "github_token": cfg["github_token"],
    }
    p = INSTALLER_TAG_PREFIX
    tags = {
        p + "source": "Data_Model_Installer",
        p + "industry": cfg["industry"],
        p + "size": cfg["model_size"],
        p + "version": version,
        p + "duration": "running",
        p + "session_id": sid,
    }
    nb_path = JobLauncher.get_current_notebook_path()
    if not nb_path:
        log("Could not determine notebook path; running the install directly instead.")
        return (False, False, None)
    job_name = "dbx_vibe_installer_%s_%s_%s" % (cfg["industry"], cfg["model_size"], version)
    log("No session_id -> launching the install as a Databricks job (%s) ..." % job_name)
    res = JobLauncher(nb_path, widgets, tags).launch(job_name=job_name, run_name=job_name)
    if not res["success"]:
        log("Job launch failed (%s) -> running the install directly instead." % res.get("error"))
        return (False, False, None)

    job_url = res["job_url"] or ("job_id=%s run_id=%s" % (res["job_id"], res["run_id"]))
    log("=" * 64)
    log("A Databricks JOB is installing %s/%s -> catalog `%s`." % (cfg["industry"], cfg["model_size"], cfg["catalog"]))
    log("Monitor it live here: %s" % job_url)
    log("Waiting for the job to finish (liveness pulse every 20s) ...")
    log("=" * 64)

    r = JobLauncher.wait_for_run(res["run_id"], job_url=job_url, pulse_seconds=20, logger=log)
    host, org = JobLauncher._get_workspace_context()
    catalog_url = ("%s/explore/data/%s%s" % (host, cfg["catalog"], ("?o=%s" % org if org else ""))) if host else ""
    nbout = r.get("notebook_output") or ""
    log("=" * 64)
    if r.get("result_state") == "SUCCESS":
        log("INSTALL JOB COMPLETE (%s)." % r.get("result_state"))
        if nbout:
            log("  Job result: %s" % nbout)
        if catalog_url:
            log("  Open the catalog here: %s" % catalog_url)
        log("=" * 64)
        exitval = "SUCCESS via job: %s/%s -> `%s` | catalog: %s | job: %s" % (
            cfg["industry"], cfg["model_size"], cfg["catalog"], catalog_url or "(n/a)", job_url)
        return (True, True, exitval)
    log("INSTALL JOB DID NOT SUCCEED (state=%s, result=%s)." % (r.get("life_cycle_state"), r.get("result_state")))
    if nbout:
        log("  Job result: %s" % nbout)
    if r.get("error"):
        log("  Job error: %s" % str(r.get("error"))[:300])
    log("  Inspect the run here: %s" % job_url)
    log("=" * 64)
    exitval = "FAILED via job: %s/%s (result=%s) | job: %s" % (
        cfg["industry"], cfg["model_size"], r.get("result_state"), job_url)
    return (True, False, exitval)


def install(cfg, plan):
    """Apply the plan phase-by-phase, retry failures serially, summarize.

    Returns (final_failures, elapsed_seconds, timings) where timings is an
    OrderedDict {phase: seconds} so the breakdown can be surfaced in the exit value."""
    from collections import OrderedDict as _OD
    run_start = time.time()
    threads, bs = cfg["threads"], cfg["batch_size"]
    catalog = cfg["catalog"]
    failures = []
    timings = _OD()

    # Metadata-mutation DDL (ADD CONSTRAINT, SET TAGS) goes through the Unity Catalog
    # API. At 32 concurrent threads UC throttles and returns 504/UC_CLIENT_EXCEPTION
    # after a few hundred calls, which collapses throughput. Cap these phases to a
    # UC-friendly concurrency; object-creation phases (table/metric) keep full threads.
    ddl_threads = cfg["ddl_threads"]

    def do(phase, stmts, bsize, group=False, serial=False, workers=None):
        t = time.time()
        w = workers if workers is not None else threads
        n = 0
        for stmt, msg in run_phase(phase, stmts, w, bsize, group=group, serial=serial):
            failures.append((phase, stmt, msg))
            n += 1
        timings[phase] = time.time() - t
        return n

    log("-" * 60)
    # Catalog is the prerequisite for every other statement. If it cannot be created
    # (e.g. a misconfigured metastore with no storage root), abort NOW with a clear
    # error instead of grinding through thousands of doomed schema/table/fk/tag calls.
    # Catalog creation is metastore-config-sensitive (Default Storage / no
    # storage root). Use the robust helper, then DO NOT run the plan's plain
    # CREATE CATALOG, which fails on Default-Storage-only metastores even when
    # the catalog already exists (the storage-root check precedes IF NOT EXISTS).
    try:
        _ensure_catalog(catalog, log)
        cat_fail = 0
    except Exception as _ce:
        failures.append(("catalog", "CREATE CATALOG `%s`" % catalog, str(_ce)))
        cat_fail = 1
    if cat_fail or not _catalog_exists(catalog):
        err = failures[-1][2] if failures else "catalog not present after CREATE"
        elapsed = time.time() - run_start
        msg = ("FATAL: catalog `%s` could not be created - aborting before the "
               "remaining %d statements. Root cause: %s"
               % (catalog, sum(len(v) for k, v in plan.items() if k != "catalog"),
                  " ".join((err or "").split())[:240]))
        log("=" * 64)
        log(msg)
        raise Exception(msg)
    schema_stmts = list(plan["schema"])
    if cfg["include_metrics"]:
        schema_stmts.append("CREATE SCHEMA IF NOT EXISTS `%s`.`_metrics`" % catalog)
    do("schema", schema_stmts, bs)
    do("table", plan["table"], bs)
    do("fk", plan["fk"], bs, group=True, workers=ddl_threads)
    do("tag", plan["tag"], bs, group=True, workers=ddl_threads)
    if cfg["include_metrics"]:
        do("metric", plan["metric"], bs)
    if plan["other"]:
        do("other", plan["other"], bs)

    log("First-pass failures (pre-retry): %d" % len(failures))
    t_retry = time.time()
    final = retry_failed(failures)
    timings["retry"] = time.time() - t_retry
    elapsed = time.time() - run_start

    log("=" * 64)
    log("INSTALL SUMMARY  %s/%s  ->  catalog `%s`" % (cfg["industry"], cfg["model_size"], catalog))
    for k, v in plan.items():
        if v:
            log("  planned %-8s %d" % (k, len(v)))
    log("  phase timings: " + "  ".join("%s=%.0fs" % (k, v) for k, v in timings.items()))
    log("  total time: %.1fs (%.1f min)" % (elapsed, elapsed / 60.0))
    log("  unrecoverable failures: %d  (enumerated below by caller)" % len(final))
    return final, elapsed, timings


# _SINK is defined in the helpers cell (next to _LOG_BUFFER) so log() can see it.


def setup_log_sink(cfg):
    """Create <catalog>._install.logs Volume and mirror the live log there every few
    seconds, so the full run log is retrievable via CLI during and after the run.
    Best effort - if it cannot be created, console/heartbeat logging still works."""
    cat = cfg["catalog"]
    try:
        _ensure_catalog(cat, log)
        spark.sql("CREATE SCHEMA IF NOT EXISTS `%s`.`_install`" % cat)
        spark.sql("CREATE VOLUME IF NOT EXISTS `%s`.`_install`.`logs`" % cat)
    except Exception as e:
        log("Log sink unavailable (%s) - console logging only" % str(e)[:160])
        return
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = "/Volumes/%s/_install/logs/install_%s_%s_%s.log" % (cat, cfg["industry"], cfg["model_size"], ts)
    # Seed the file with everything logged so far, then switch log() to inline append.
    try:
        with _LOG_LOCK:
            seed = "\n".join(_LOG_BUFFER)
            with open(path, "w") as f:
                f.write(seed + "\n")
            _SINK["path"] = path
    except Exception as e:
        log("Log sink unavailable (%s) - console logging only" % str(e)[:160])
        return
    log("Log sink: %s" % path)


def teardown_log_sink():
    # Inline append already persisted every line; nothing to flush.
    _SINK["path"] = None


def write_failures_manifest(cfg, final):
    """Persist unrecoverable statements (phase, error, SQL) to the install volume so a
    user can review/replay them after the run. Best effort."""
    cat = cfg["catalog"]
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = "/Volumes/%s/_install/logs/failures_%s_%s_%s.json" % (cat, cfg["industry"], cfg["model_size"], ts)
    try:
        rows = [{"phase": ph, "error": err, "sql": stmt} for ph, stmt, err in final]
        with open(path, "w") as f:
            f.write(json.dumps(rows, indent=2))
            f.flush()
            os.fsync(f.fileno())   # durable before dbutils.notebook.exit() terminates us
        log("Failures manifest: %s" % path)
    except Exception as e:
        log("Could not write failures manifest (%s)" % str(e)[:160])


def main():
    # First-run guard: an industry must be chosen explicitly. The widget defaults to the
    # SELECT_PROMPT placeholder, so on the very first Run All nothing is selected yet -
    # print a friendly prompt and stop cleanly (no error, no job launched).
    if _wget("model", "").strip() not in INDUSTRIES:
        log("Please select an industry in the '1. industry' widget, then click Run All again.")
        return

    cfg = resolve_config()
    log("Mode          : %s" % cfg["mode"])
    log("Industry      : %s" % cfg["industry"])
    log("Model size    : %s" % cfg["model_size"])
    log("Target catalog: %s" % cfg["catalog"])
    log("Threads       : %d   Batch size: %d" % (cfg["threads"], cfg["batch_size"]))
    log("Metric views  : %s" % cfg["include_metrics"])

    # Launcher gate: an interactive run with no session_id launches the install as a
    # Databricks job. The _running_as_job() backstop guarantees a launched job runs the
    # install in-place instead of re-launching itself.
    if not cfg["session_id"] and not _running_as_job():
        handled, ok, exitval = launch_and_wait(cfg)
        if handled:
            dbutils.notebook.exit(exitval)
            return

    # Running in-place (the launched job, or launch was unavailable). Keep the log
    # sink OPEN through the final verdict + manifest so they land in the Volume log;
    # _flush_log_durable() fsyncs the last state before exit, then teardown.
    setup_log_sink(cfg)
    result = None
    try:
        plan = build_plan(cfg)
        final, elapsed, timings = install(cfg, plan)

        # Tag the job with the final install duration (best effort; only when run as a job).
        p = INSTALLER_TAG_PREFIX
        tag_res = JobLauncher.update_job_tags({
            p + "industry": cfg["industry"],
            p + "size": cfg["model_size"],
            p + "version": cfg.get("resolved_version", "unknown"),
            p + "duration": "%.1fmin" % (elapsed / 60.0),
        })
        log("Tag update: %s" % ("ok" if tag_res["success"] else tag_res.get("error")))

        timing_str = " ".join("%s=%.0fs" % (k, v) for k, v in timings.items())
        total = sum(len(v) for v in plan.values())

        if final:
            # Structural phases must be perfect; metric views are source-model SQL and a
            # bad column reference there is the model author's defect, not an install
            # fault. So: hard-fail on any structural failure, warn on metric-only ones.
            structural = [f for f in final if f[0] != "metric"]
            log("=" * 64)
            log("UNRECOVERABLE STATEMENTS: %d  (structural=%d, metric=%d)"
                % (len(final), len(structural), len(final) - len(structural)))
            for phase, stmt, err in final:
                log("  [%s] %s" % (phase, " ".join(stmt.split())[:110]))
                log("        -> %s" % (" ".join((err or "").split())[:200]))
            write_failures_manifest(cfg, final)   # persisted to <catalog>._install volume
            if structural:
                raise Exception(
                    "Install FAILED: %d structural statement(s) unrecoverable in `%s` "
                    "(+%d metric-view source defects) | timings: %s"
                    % (len(structural), cfg["catalog"], len(final) - len(structural), timing_str))
            result = ("INSTALLED_WITH_WARNINGS: %s/%s -> `%s` (%d statements, %d metric-view "
                      "source defects, 0 structural failures, %.1f min) | timings: %s | log: %s"
                      % (cfg["industry"], cfg["model_size"], cfg["catalog"], total,
                         len(final), elapsed / 60.0, timing_str, _SINK["path"]))
            log(result)
        else:
            result = ("SUCCESS: %s/%s -> `%s` (%d statements, 0 failures, %.1f min) | timings: %s | log: %s"
                      % (cfg["industry"], cfg["model_size"], cfg["catalog"],
                         total, elapsed / 60.0, timing_str, _SINK["path"]))
            log(result)
    finally:
        _flush_log_durable()   # fsync the complete log (incl. final verdict) before exit
        teardown_log_sink()

    dbutils.notebook.exit(result)


main()
